In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

login(secret_value_0 )

print("Hugging face Authenticated Successfully")

Hugging face Authenticated Successfully


In [4]:

%pip install -q datasets pandas


Note: you may need to restart the kernel to use updated packages.


In [5]:

from datasets import load_dataset
from collections import Counter

DATASET_ID = "hypaai/Hypa-Text-10k"

dataset = load_dataset(DATASET_ID, split="train")

print("Dataset:", DATASET_ID)
print("Rows:", len(dataset))
print("Columns:", dataset.column_names)
print("Features:", dataset.features)

# Inspect the structure without printing the entire dataset.
sample = dataset[0]
print("\nFirst record keys:", list(sample.keys()))

for message in sample.get("messages", []):
    print({
        "role": message.get("role"),
        "content_preview": message.get("content", "")[:250],
    })


README.md:   0%|          | 0.00/18.8k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.32MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset: hypaai/Hypa-Text-10k
Rows: 10000
Columns: ['messages', 'tools_json']
Features: {'messages': List({'role': Value('string'), 'content': Value('string'), 'thinking': Value('string'), 'tool_calls': List({'function': {'name': Value('string'), 'arguments': Value('string')}}), 'tool_responses': List({'name': Value('string'), 'response': Value('string')})}), 'tools_json': Value('string')}

First record keys: ['messages', 'tools_json']
{'role': 'system', 'content_preview': 'You are Hypa Translate. Translate from Urhobo to English. Return only the exact translation.\n\nTHINKING=None'}
{'role': 'user', 'content_preview': 'Ruth rọvwọn re, o ji vwo emọ ẹnẹ.'}
{'role': 'assistant', 'content_preview': 'Ruth is married with four children.'}


In [6]:

import re
from collections import Counter

system_prompt_counts = Counter()
candidate_rows = []

for index, row in enumerate(dataset):
    messages = row.get("messages", [])

    system_messages = [
        message.get("content", "")
        for message in messages
        if message.get("role") == "system"
    ]

    system_text = "\n".join(system_messages)

    # Identify records whose instructions mention both languages.
    if (
        "english" in system_text.lower()
        and "igala" in system_text.lower()
        and "translat" in system_text.lower()
    ):
        system_prompt_counts[system_text] += 1
        candidate_rows.append((index, row))

print("Records mentioning English, Igala, and translation:",
      len(candidate_rows))

print("\nMatching system-prompt variants:")
for prompt, count in system_prompt_counts.most_common():
    print(f"\nCount: {count}")
    print(prompt[:500])

print("\nSample candidate records:")
for index, row in candidate_rows[:5]:
    print(f"\nDataset index: {index}")

    for message in row.get("messages", []):
        if message.get("role") in {"user", "assistant"}:
            print(
                message["role"],
                ":",
                message.get("content", "")[:400]
            )


Records mentioning English, Igala, and translation: 102

Matching system-prompt variants:

Count: 41
You are Hypa Translate. Translate from English to Igala. Return only the exact translation.

THINKING=None

Count: 36
You are Hypa Translate. Translate from Igala to English. Return only the exact translation.

THINKING=None

Count: 25
You are Hypa Translate. Translate from English to Igala. Preserve meaning faithfully and output only the final translation with no explanation, no notes, and no extra text.

THINKING=None

Sample candidate records:

Dataset index: 74
user : To the south across Bohicket Creek is the town of Seabrook Island.
assistant : Teju'ọpata ọganẹ udaba Bohicket Creek oñwu chefewou lẹ, Seabrook Island.

Dataset index: 77
user : Their mascot is the Bulsajo, which translates to "Phoenix" in English.
assistant : Ewñ ku ma du té ki k'oji mashi the Bulsajo é ki mu di phoenix efi alu ènefu.

Dataset index: 87
user : Earlier in his career he was told that he could have been 

In [ ]:

from collections import Counter, defaultdict

prompt_counts = Counter()
examples_by_prompt = defaultdict(list)

for index, row in enumerate(dataset):
    messages = row.get("messages", [])

    system_text = "\n".join(
        message.get("content", "")
        for message in messages
        if message.get("role") == "system"
    )

    normalized = system_text.lower()

    if "igala" in normalized and "translat" in normalized:
        prompt_counts[system_text] += 1
        examples_by_prompt[system_text].append(index)

print("Total records mentioning Igala and translation:",
      sum(prompt_counts.values()))

print("\nAll matching prompt variants:")
for prompt, count in prompt_counts.most_common():
    print(f"\nCount: {count}")
    print(prompt[:350])


In [8]:

import re
from collections import Counter

direction_counts = Counter()
candidate_records = []

for index, row in enumerate(dataset):
    messages = row.get("messages", [])

    system_text = "\n".join(
        message.get("content", "")
        for message in messages
        if message.get("role") == "system"
    ).lower()

    user_messages = [
        m.get("content", "").strip()
        for m in messages
        if m.get("role") == "user"
    ]

    assistant_messages = [
        m.get("content", "").strip()
        for m in messages
        if m.get("role") == "assistant"
    ]

    if "igala" not in system_text or "translat" not in system_text:
        continue

    if "from english to igala" in system_text:
        direction = "english_to_igala"
    elif "from igala to english" in system_text:
        direction = "igala_to_english"
    elif re.search(r"translate to igala", system_text):
        direction = "target_only_or_unspecified"
    else:
        direction = "other_igala_translation"

    direction_counts[direction] += 1

    candidate_records.append({
        "index": index,
        "direction": direction,
        "system": system_text,
        "user": "\n".join(user_messages),
        "assistant": "\n".join(assistant_messages),
        "user_count": len(user_messages),
        "assistant_count": len(assistant_messages),
    })

print("Direction counts:")
for direction, count in direction_counts.items():
    print(f"{direction}: {count}")

print("\nEnglish-to-Igala examples:")
shown = 0

for record in candidate_records:
    if record["direction"] != "english_to_igala":
        continue

    print(f"\nIndex: {record['index']}")
    print("User:", record["user"][:250])
    print("Assistant:", record["assistant"][:250])

    shown += 1
    if shown == 10:
        break


Direction counts:
other_igala_translation: 975
target_only_or_unspecified: 64
english_to_igala: 66
igala_to_english: 36

English-to-Igala examples:

Index: 74
User: To the south across Bohicket Creek is the town of Seabrook Island.
Assistant: Teju'ọpata ọganẹ udaba Bohicket Creek oñwu chefewou lẹ, Seabrook Island.

Index: 77
User: Their mascot is the Bulsajo, which translates to "Phoenix" in English.
Assistant: Ewñ ku ma du té ki k'oji mashi the Bulsajo é ki mu di phoenix efi alu ènefu.

Index: 87
User: Earlier in his career he was told that he could have been a sprinter.
Assistant: Iko ukòchè wu makaki I'ma chi ènè kiya rule.

Index: 111
User: Glad you're back at last.
Assistant: Ẹdẹbò iche uwè mudakubiwa otitala.

Index: 137
User: The silver anchor on a blue background symbolizes the importance of the local harbor.
Assistant: Agbo un̄yanẹ ajifa oji odufa bulu ejubi lẹ du n̄a ogbogaga ugbo ki ami ọkọ eju omi amanẹ.

Index: 648
User: Following is a sortable list of museums in Egypt.
As

In [9]:

english_to_igala = [
    r for r in candidate_records
    if r["direction"] == "english_to_igala"
]

empty_records = [
    r for r in english_to_igala
    if not r["user"] or not r["assistant"]
]

source_counts = Counter(
    r["user"] for r in english_to_igala
)

pair_counts = Counter(
    (r["user"], r["assistant"])
    for r in english_to_igala
)

print("Explicit English-to-Igala rows:", len(english_to_igala))
print("Rows with empty source or target:", len(empty_records))
print("Unique source texts:", len(source_counts))
print("Unique source-target pairs:", len(pair_counts))
print("Repeated source texts:",
      sum(count > 1 for count in source_counts.values()))
print("Repeated source-target pairs:",
      sum(count > 1 for count in pair_counts.values()))


Explicit English-to-Igala rows: 66
Rows with empty source or target: 0
Unique source texts: 66
Unique source-target pairs: 66
Repeated source texts: 0
Repeated source-target pairs: 0


In [10]:

from collections import Counter

target_only_records = []
generic_records = []

for index, row in enumerate(dataset):
    messages = row.get("messages", [])

    system_text = "\n".join(
        m.get("content", "")
        for m in messages
        if m.get("role") == "system"
    ).strip()

    normalized = system_text.lower()

    if "igala" not in normalized or "translat" not in normalized:
        continue

    if (
        "translate to igala" in normalized
        and "from " not in normalized
    ):
        category = "target_only"
    elif (
        "specialized in translating to igala" in normalized
    ):
        category = "generic_specialist"
    else:
        continue

    user_text = "\n".join(
        m.get("content", "")
        for m in messages
        if m.get("role") == "user"
    ).strip()

    assistant_text = "\n".join(
        m.get("content", "")
        for m in messages
        if m.get("role") == "assistant"
    ).strip()

    record = {
        "index": index,
        "category": category,
        "system": system_text,
        "source": user_text,
        "target": assistant_text,
    }

    if category == "target_only":
        target_only_records.append(record)
    else:
        generic_records.append(record)

print("Target-only records:", len(target_only_records))
print("Generic specialist records:", len(generic_records))

for category, records in [
    ("TARGET-ONLY", target_only_records),
    ("GENERIC SPECIALIST", generic_records),
]:
    print(f"\n{'=' * 25} {category} {'=' * 25}")

    for record in records:
        print(f"\nIndex: {record['index']}")
        print("Instruction:", record["system"].splitlines()[0])
        print("Source:", record["source"][:200])
        print("Target:", record["target"][:200])


Target-only records: 64
Generic specialist records: 7

========================= TARGET-ONLY =========================

Index: 70
Instruction: You are Hypa Translate. Translate to Igala. Return only the exact translation.
Source: They succeeded for the first time in achieving a closed shop at Bethlehem.
Target: Mafuche ifojinẹ ọgba-ejodudu kuma n'unyi ẹñwù eta efù Betlehem.

Index: 145
Instruction: You are Hypa Translate. Translate to Igala. Return only the exact translation.
Source: Williams Township was named for a family of pioneer settlers.
Target: Williams Township chodu abo akwọr'unyi ku ma dodo ọmọ ejodudu.

Index: 434
Instruction: You are Hypa Translate. Translate to Igala. Return only the exact translation.
Source: Typically two or more colours are incorporated in the sash, in vertical stripes.
Target: Abalé amo'odö méji baki étá máche dama éfi úgwoyú lé éfi okpãlá ikú.

Index: 688
Instruction: You are Hypa Translate. Translate to Igala. Return only the exact translation.
Sour

In [11]:
# combine and audit the candidate pairs


import re
import hashlib
from collections import Counter

candidate_records = []

for index, row in enumerate(dataset):
    messages = row.get("messages", [])

    system_text = "\n".join(
        str(m.get("content", ""))
        for m in messages
        if m.get("role") == "system"
    ).strip()

    normalized = system_text.lower()

    if "igala" not in normalized or "translat" not in normalized:
        continue

    if (
        "english" in normalized
        and "igala" in normalized
        and (
            "from english to igala" in normalized
            or "english-to-igala" in normalized
            or "english → igala" in normalized
        )
    ):
        category = "explicit_english_to_igala"

    elif (
        "translate to igala" in normalized
        and "from " not in normalized
    ):
        category = "target_only"

    elif "specialized in translating to igala" in normalized:
        category = "generic_specialist"

    else:
        continue

    source = "\n".join(
        str(m.get("content", ""))
        for m in messages
        if m.get("role") == "user"
    ).strip()

    target = "\n".join(
        str(m.get("content", ""))
        for m in messages
        if m.get("role") == "assistant"
    ).strip()

    # Remove an outer translation instruction and quoted source text
    # when present in the user message.
    source = re.sub(
        r"^\s*Translate this to Igala and Return ONLY THE EXACT "
        r"TRANSLATION\.\s*",
        "",
        source,
        flags=re.IGNORECASE,
    ).strip()

    if len(source) >= 2 and source[0] == source[-1] == '"':
        source = source[1:-1].strip()

    candidate_records.append({
        "source_index": index,
        "category": category,
        "input": source,
        "target": target,
        "language_code": "igl",
    })

print("Candidate counts:")
for category, count in Counter(
    record["category"] for record in candidate_records
).items():
    print(f"  {category}: {count}")

print("\nTotal candidates:", len(candidate_records))

valid_records = [
    record for record in candidate_records
    if record["input"].strip() and record["target"].strip()
]

print("Non-empty pairs:", len(valid_records))
print("Empty pairs removed:", len(candidate_records) - len(valid_records))

normalized_sources = [
    re.sub(r"\s+", " ", r["input"].strip()).casefold()
    for r in valid_records
]

normalized_pairs = [
    (
        re.sub(r"\s+", " ", r["input"].strip()).casefold(),
        re.sub(r"\s+", " ", r["target"].strip()).casefold(),
    )
    for r in valid_records
]

print("Unique source texts:", len(set(normalized_sources)))
print("Unique source-target pairs:", len(set(normalized_pairs)))
print("Duplicate source occurrences:", len(normalized_sources) - len(set(normalized_sources)))
print("Duplicate pair occurrences:", len(normalized_pairs) - len(set(normalized_pairs)))

print("\nLength summary (characters):")
print("  Median source length:", sorted(
    len(r["input"]) for r in valid_records
)[len(valid_records) // 2] if valid_records else 0)
print("  Median target length:", sorted(
    len(r["target"]) for r in valid_records
)[len(valid_records) // 2] if valid_records else 0)

print("\nSample pairs for manual review:")
for record in valid_records[:10]:
    print(f"\nIndex: {record['source_index']}")
    print("Category:", record["category"])
    print("English:", record["input"][:250])
    print("Igala:", record["target"][:250])


Candidate counts:
  target_only: 64
  explicit_english_to_igala: 66
  generic_specialist: 7

Total candidates: 137
Non-empty pairs: 137
Empty pairs removed: 0
Unique source texts: 137
Unique source-target pairs: 137
Duplicate source occurrences: 0
Duplicate pair occurrences: 0

Length summary (characters):
  Median source length: 63
  Median target length: 56

Sample pairs for manual review:

Index: 70
Category: target_only
English: They succeeded for the first time in achieving a closed shop at Bethlehem.
Igala: Mafuche ifojinẹ ọgba-ejodudu kuma n'unyi ẹñwù eta efù Betlehem.

Index: 74
Category: explicit_english_to_igala
English: To the south across Bohicket Creek is the town of Seabrook Island.
Igala: Teju'ọpata ọganẹ udaba Bohicket Creek oñwu chefewou lẹ, Seabrook Island.

Index: 77
Category: explicit_english_to_igala
English: Their mascot is the Bulsajo, which translates to "Phoenix" in English.
Igala: Ewñ ku ma du té ki k'oji mashi the Bulsajo é ki mu di phoenix efi alu ènefu.

In

In [1]:

from datasets import load_dataset
import pandas as pd
import re

DATASET_ID = "hypaai/Hypa_AIME2024"

# Load only the English and Igala splits.
english_ds = load_dataset(DATASET_ID, split="english")
igala_ds = load_dataset(DATASET_ID, split="igala")

print("English rows:", len(english_ds))
print("Igala rows:", len(igala_ds))

print("\nEnglish columns:", english_ds.column_names)
print("Igala columns:", igala_ds.column_names)

assert len(english_ds) == len(igala_ds), (
    "English and Igala splits have different lengths. "
    "Do not align records by row position until investigated."
)

def get_problem(row):
    # The dataset card has used both field names in its description.
    for key in ("problem", "question"):
        if key in row and row[key] is not None:
            return str(row[key]).strip()
    raise KeyError(f"No problem/question field found: {row.keys()}")

def normalize_math_text(text):
    return re.sub(r"\s+", " ", text).strip()

english_problems = [get_problem(row) for row in english_ds]
igala_problems = [get_problem(row) for row in igala_ds]

# The answer fields should match because the mathematical problem
# and its answer are intended to remain aligned across languages.
english_answers = [
    str(row.get("answer", "")).strip()
    for row in english_ds
]
igala_answers = [
    str(row.get("answer", "")).strip()
    for row in igala_ds
]

answer_mismatches = [
    i for i, (en, ig) in enumerate(
        zip(english_answers, igala_answers)
    )
    if en != ig
]

pairs = [
    {
        "id": f"aime2024-{i + 1:02d}",
        "language_code": "igl",
        "input": english_problems[i],
        "target": igala_problems[i],
        "answer": english_answers[i],
    }
    for i in range(len(english_ds))
]

empty_pairs = [
    p["id"] for p in pairs
    if not p["input"] or not p["target"]
]

duplicate_english = (
    pd.Series([
        normalize_math_text(p["input"]).casefold()
        for p in pairs
    ]).duplicated().sum()
)

print("\n--- Alignment audit ---")
print("Answer mismatches:", len(answer_mismatches))
print("Empty pairs:", len(empty_pairs))
print("Duplicate English questions:", int(duplicate_english))

print("\n--- Sample aligned pairs ---")
for p in pairs[:5]:
    print(f"\n{p['id']}")
    print("English:", p["input"][:500])
    print("Igala:", p["target"][:500])
    print("Answer:", p["answer"])

print("\n--- Audit decision ---")
if answer_mismatches or empty_pairs:
    print("REVIEW REQUIRED: alignment or completeness issue detected.")
else:
    print(
        "STRUCTURAL CHECK PASSED. "
        "Native-speaker review and leakage checks are still required."
    )


README.md:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

data/english-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 8.77kB            

data/english-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/annang-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.14kB            

data/annang-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating ibibio split:   0%|          | 0/30 [00:00<?, ? examples/s]

Generating igbo split:   0%|          | 0/30 [00:00<?, ? examples/s]

Generating tiv split:   0%|          | 0/30 [00:00<?, ? examples/s]

Generating yoruba split:   0%|          | 0/30 [00:00<?, ? examples/s]

English rows: 30
Igala rows: 30

English columns: ['problem', 'answer']
Igala columns: ['problem', 'answer']

--- Alignment audit ---
Answer mismatches: 0
Empty pairs: 0
Duplicate English questions: 0

--- Sample aligned pairs ---

aime2024-01
English: Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: 
 \[\log_2\left({x \over yz}\right) = {1 \over 2}\]
 \[\log_2\left({y \over xz}\right) = {1 \over 3}\]
 \[\log_2\left({z \over xy}\right) = {1 \over 4}\]
 Then the value of $\left|\log_2(x^4y^3z^2)\right|$ is $\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$.
Igala: Jé $x,y$ kpài $z$ kísho oshóshí ámí aluká ki bòjò systemu lé kidogbá kishéjuká: 
 \[\log_2\left({x \over yz}\right) = {1 \over 2}\]
 \[\log_2\left({y \over xz}\right) = {1 \over 3}\]
 \[\log_2\left({z \over xy}\right) = {1 \over 4}\]
 Takí ínyí $\left|\log_2(x^4y^3z^2)\right|$ is $\tfrac{m}{n}$ ungbó $m$ kpài $n$ shé biolá éjodudu oshóshí ámí aluká ényó

In [ ]:

import re
from difflib import SequenceMatcher

def normalize_text(text):
    text = str(text or "").casefold()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

aime_sources = {
    normalize_text(record["input"]): record
    for record in pairs
}

hypa_sources = {
    normalize_text(record["input"]): record
    for record in candidate_records
    if record.get("input", "").strip()
}

# Exact normalized-source overlap
exact_overlap = sorted(
    set(aime_sources) & set(hypa_sources)
)

print("AIME examples:", len(pairs))
print("Hypa candidates:", len(candidate_records))
print("Exact source overlaps:", len(exact_overlap))

for source in exact_overlap:
    print("\nOverlapping source:")
    print(source[:300])

# Near-duplicate screening.
# This is a review aid, not proof that two questions are equivalent.
hypa_texts = list(hypa_sources.keys())
near_matches = []

for aime_text, aime_record in aime_sources.items():
    for hypa_text in hypa_texts:
        if aime_text == hypa_text:
            continue

        similarity = SequenceMatcher(
            None, aime_text, hypa_text
        ).ratio()

        if similarity >= 0.85:
            near_matches.append({
                "aime_id": aime_record.get("id", "unknown"),
                "similarity": round(similarity, 3),
                "aime_source": aime_text[:180],
                "hypa_source": hypa_text[:180],
            })

print("\nPotential near-duplicate pairs:", len(near_matches))

for match in near_matches[:20]:
    print("\n", match)
